<div dir="rtl" align="right">

# مدل مخلوط گاوسی (GMM)

درسی از مدرسه پایتون و هوش مصنوعی، دکتر بهروز طاهری

## ایده در یک نگاه
فرض می‌کنیم داده از **چند توزیع نرمال (گاوسی)** آمده است؛ هر خوشه یک گاوسی با میانگین $\mu_k$، ماتریس کوواریانس $\Sigma_k$ (شکل و جهت بیضی) و وزن $\pi_k$:

$$p(\mathbf x)=\sum_{k=1}^{K}\pi_k\,\mathcal N(\mathbf x\mid\mu_k,\Sigma_k)$$

برخلاف K-Means که هر نقطه را **قطعاً** به یک خوشه می‌دهد، GMM **احتمال عضویت** می‌دهد (خوشه‌بندی نرم)؛ و خوشه‌ها می‌توانند **بیضوی و کشیده** باشند.

> [K-Means خوشه‌های گرد و برچسب قطعی؛ GMM بیضی‌های کشیده و رنگ‌های ترکیبی (عضویت نرم).](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/gmm.png)

*K-Means خوشه‌های گرد و برچسب قطعی؛ GMM بیضی‌های کشیده و رنگ‌های ترکیبی (عضویت نرم).*

## الگوریتم EM
دو گام تکرار می‌شوند:
- **E (امید):** برای هر نقطه احتمال عضویت در هر مؤلفه: $r_{ik}=\frac{\pi_k\mathcal N(x_i\mid\mu_k,\Sigma_k)}{\sum_j\pi_j\mathcal N(x_i\mid\mu_j,\Sigma_j)}$
- **M (بیشینه‌سازی):** با این وزن‌ها پارامترها را دوباره حساب کن: $\mu_k=\frac{\sum_ir_{ik}x_i}{\sum_ir_{ik}}$، $\pi_k=\frac1n\sum_ir_{ik}$ و کوواریانس وزن‌دار.

هر دور EM، **درست‌نمایی** را کم نمی‌کند؛ پس همگرا می‌شود (ولی شاید به بهینه محلی).

[[lab:gmm]]

در آزمایشگاه «یک گام EM» را چند بار بزن و ببین بیضی‌ها چطور جای خود را پیدا می‌کنند و log-likelihood بالا می‌رود.

## مثال عددی یک گام E (یک‌بعدی)
دو مؤلفه با $\mu_1=0$، $\mu_2=4$، $\sigma=1$ و $\pi=0.5$. برای $x=1$:

| | چگالی $\mathcal N(1\mid\mu,1)$ | عضویت |
|---|---|---|
| مؤلفه ۱ | 0.242 | 0.242/(0.242+0.0044) = **0.982** |
| مؤلفه ۲ | 0.0044 | **0.018** |



</div>

In [ ]:
import numpy as np
from scipy.stats import norm

x = 1.0
d1, d2 = norm.pdf(x, 0, 1), norm.pdf(x, 4, 1)
r1 = 0.5 * d1 / (0.5 * d1 + 0.5 * d2)
print(f"densities: {d1:.4f}, {d2:.4f}")
print(f"membership: component 1 = {r1:.3f}, component 2 = {1 - r1:.3f}")

<div dir="rtl" align="right">



## انتخاب تعداد مؤلفه: BIC
$\text{BIC}=-2\ln L+p\ln n$؛ $p$ تعداد پارامترهاست. مدلی با **کمترین BIC** را انتخاب کن؛ جمله دوم مدل پیچیده را جریمه می‌کند.

| `covariance_type` | شکل خوشه |
|---|---|
| `full` | هر بیضی با هر جهت (پیش‌فرض) |
| `tied` | همه یک شکل |
| `diag` | بیضی موازی محورها |
| `spherical` | دایره (شبیه K-Means) |



</div>

In [ ]:
import numpy as np
from sklearn.mixture import GaussianMixture

rng = np.random.default_rng(0)
X = np.r_[rng.multivariate_normal([0, 0], [[3, 2.2], [2.2, 2]], 200),
          rng.multivariate_normal([5, 0], [[0.5, 0], [0, 2]], 200),
          rng.multivariate_normal([2, 5], [[1, 0], [0, 0.4]], 200)]
for k in range(1, 7):
    g = GaussianMixture(k, random_state=0).fit(X)
    print(f"K={k}  BIC={g.bic(X):8.1f}")
best = GaussianMixture(3, random_state=0).fit(X)
print("weights:", best.weights_.round(3))
print("soft labels of 3 points:\n", best.predict_proba(X[[0, 250, 500]]).round(3))

<div dir="rtl" align="right">



> GMM یک **مدل مولد** است: با `sample()` می‌توانی داده جدید شبیه داده اصلی بسازی؛ و با `score_samples` نقاط کم‌احتمال (ناهنجار) را پیدا کنی.

</div>